# 02 — Pré-processamento e Validação do Pipeline

**Objetivo:** Verificar a integridade das imagens, validar as transformações e calcular
as estatísticas do dataset (média/desvio) para normalização personalizada, se necessário.

Ao final deste notebook teremos:
- Relatório de imagens corrompidas (se houver)
- Visualização das augmentações aplicadas
- Confirmação de que os DataLoaders funcionam corretamente
- Análise do desbalanceamento e estratégia de mitigação

## 1. Configuração

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('..') / 'src'))

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from PIL import Image
from tqdm.notebook import tqdm

from config import (
    CAMINHO_TREINO, CAMINHO_TESTE, CAMINHO_VALIDACAO,
    NOMES_CLASSES, TAMANHO_LOTE, SEMENTE_ALEATORIA,
    MEDIA_IMAGENET, DESVIO_IMAGENET,
)
from dataset import DatasetPulmao, obter_transformacoes, criar_dataloaders

plt.rcParams['figure.dpi'] = 120
sns.set_theme(style='whitegrid')
torch.manual_seed(SEMENTE_ALEATORIA)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')

## 2. Verificação de Integridade das Imagens

In [ ]:
def verificar_imagens(caminho_raiz: Path) -> list[str]:
    """
    Tenta abrir cada imagem e registra as corrompidas.
    Retorna lista de caminhos com problema.
    """
    corrompidas = []
    arquivos = list(caminho_raiz.rglob('*.png')) + list(caminho_raiz.rglob('*.jpg'))

    for arquivo in tqdm(arquivos, desc=f'Verificando {caminho_raiz.name}'):
        try:
            with Image.open(arquivo) as img:
                img.verify()  # Verifica a integridade do arquivo sem decodificar
        except Exception as e:
            corrompidas.append(str(arquivo))
            print(f'CORROMPIDA: {arquivo.name} — {e}')

    return corrompidas

# Verifica os três splits
corrompidas_treino = verificar_imagens(CAMINHO_TREINO)
corrompidas_valid  = verificar_imagens(CAMINHO_VALIDACAO)
corrompidas_teste  = verificar_imagens(CAMINHO_TESTE)

total_corrompidas = len(corrompidas_treino) + len(corrompidas_valid) + len(corrompidas_teste)
print(f'\nTotal de imagens corrompidas: {total_corrompidas}')
if total_corrompidas == 0:
    print('Todos os arquivos estão íntegros!')

## 3. Visualização das Augmentações

In [ ]:
def desnormalizar(tensor: torch.Tensor) -> np.ndarray:
    """
    Reverte a normalização ImageNet para exibição visual.
    Entrada: tensor (C, H, W). Saída: array (H, W, C) com valores [0, 1].
    """
    media  = torch.tensor(MEDIA_IMAGENET).view(3, 1, 1)
    desvio = torch.tensor(DESVIO_IMAGENET).view(3, 1, 1)
    img = tensor * desvio + media
    img = img.permute(1, 2, 0).numpy()
    return np.clip(img, 0, 1)


# Carrega o dataset de treino (com augmentações)
dataset_treino = DatasetPulmao(
    caminho_raiz=CAMINHO_TREINO,
    transformacoes=obter_transformacoes('treino'),
)

# Seleciona uma imagem de cada classe
indices_por_classe = {}
for idx, rotulo in enumerate(dataset_treino.rotulos):
    if rotulo not in indices_por_classe:
        indices_por_classe[rotulo] = idx
    if len(indices_por_classe) == 4:
        break

print(f'Dataset de treino: {len(dataset_treino)} imagens')
print('Índices por classe:', indices_por_classe)

In [ ]:
# Mostra 5 versões augmentadas da mesma imagem
N_AUGMENTACOES = 5
N_CLASSES = len(indices_por_classe)

fig, axes = plt.subplots(N_CLASSES, N_AUGMENTACOES + 1, figsize=(14, N_CLASSES * 2.8))

for linha_idx, (rotulo, idx_imagem) in enumerate(sorted(indices_por_classe.items())):
    # Imagem original (sem transformação)
    caminho_original = dataset_treino.caminhos_imagens[idx_imagem]
    img_original = Image.open(caminho_original).convert('RGB').resize((224, 224))

    axes[linha_idx, 0].imshow(img_original)
    axes[linha_idx, 0].axis('off')
    axes[linha_idx, 0].set_title('Original', fontsize=8)
    if linha_idx == 0:
        axes[linha_idx, 0].set_title('Original', fontsize=9, fontweight='bold')

    # Versões augmentadas
    for col in range(1, N_AUGMENTACOES + 1):
        tensor_aug, _ = dataset_treino[idx_imagem]
        img_aug = desnormalizar(tensor_aug)
        axes[linha_idx, col].imshow(img_aug)
        axes[linha_idx, col].axis('off')
        if linha_idx == 0:
            axes[linha_idx, col].set_title(f'Aug {col}', fontsize=9)

    # Rótulo da linha
    axes[linha_idx, 0].set_ylabel(NOMES_CLASSES[rotulo].split(' ')[0],
                                   rotation=0, labelpad=60,
                                   va='center', fontsize=9)

plt.suptitle('Data Augmentation — Amostras Geradas por Classe', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/exemplos_augmentacao.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Verificação dos DataLoaders

In [ ]:
# Instancia os DataLoaders
loader_treino, loader_valid, loader_teste = criar_dataloaders(
    tamanho_lote=TAMANHO_LOTE,
    num_workers=0,  # 0 para evitar erros em notebooks no macOS
)

# Verifica um batch de treino
imagens_batch, rotulos_batch = next(iter(loader_treino))

print(f'Shape do batch de imagens : {imagens_batch.shape}')
print(f'Shape dos rótulos         : {rotulos_batch.shape}')
print(f'Dtype das imagens         : {imagens_batch.dtype}')
print(f'Min/Max valores           : {imagens_batch.min():.3f} / {imagens_batch.max():.3f}')
print(f'Rótulos no batch          : {rotulos_batch.tolist()}')

## 5. Cálculo das Estatísticas do Dataset (Opcional)

In [ ]:
def calcular_media_desvio(loader: torch.utils.data.DataLoader) -> tuple:
    """
    Calcula a média e o desvio padrão por canal RGB do dataset de treino.
    Útil para substituir as estatísticas ImageNet por estatísticas próprias.
    """
    soma         = torch.zeros(3)
    soma_sq      = torch.zeros(3)
    n_pixels     = 0

    for imgs, _ in tqdm(loader, desc='Calculando estatísticas'):
        # imgs: (N, C, H, W) — soma ao longo de N, H, W
        soma     += imgs.sum(dim=(0, 2, 3))
        soma_sq  += (imgs ** 2).sum(dim=(0, 2, 3))
        n_pixels += imgs.shape[0] * imgs.shape[2] * imgs.shape[3]

    media  = soma / n_pixels
    desvio = (soma_sq / n_pixels - media ** 2).sqrt()
    return media.tolist(), desvio.tolist()

# NOTA: Use um loader SEM normalização para calcular as estatísticas reais
import torchvision.transforms as T
from dataset import DatasetPulmao

transform_sem_norm = T.Compose([T.Resize((224, 224)), T.ToTensor()])
ds_sem_norm = DatasetPulmao(CAMINHO_TREINO, transformacoes=transform_sem_norm)
loader_sem_norm = torch.utils.data.DataLoader(ds_sem_norm, batch_size=32, num_workers=0)

media_dataset, desvio_dataset = calcular_media_desvio(loader_sem_norm)

print(f'\nEstatísticas do dataset de treino:')
print(f'  Média  : {[round(v, 4) for v in media_dataset]}')
print(f'  Desvio : {[round(v, 4) for v in desvio_dataset]}')
print(f'\nEstatísticas ImageNet (referência):')
print(f'  Média  : {MEDIA_IMAGENET}')
print(f'  Desvio : {DESVIO_IMAGENET}')
print('\nComo o dataset é pequeno, usaremos as estatísticas ImageNet (recomendado para Transfer Learning).')

## 6. Análise do Desbalanceamento e Estratégia


In [ ]:
from collections import Counter
import pandas as pd

# Conta os rótulos no treino
contagem = Counter(dataset_treino.rotulos)
df_contagem = pd.DataFrame([
    {'Classe': NOMES_CLASSES[k], 'Amostras': v, 'Peso sugerido': round(max(contagem.values()) / v, 3)}
    for k, v in sorted(contagem.items())
])

print('Contagem e pesos sugeridos para a função de perda ponderada:')
display(df_contagem)

# Calcula pesos de classe para CrossEntropyLoss
total = sum(contagem.values())
pesos_classe = torch.tensor([
    total / (len(contagem) * contagem[i]) for i in sorted(contagem.keys())
], dtype=torch.float32)

print(f'\nTensor de pesos para CrossEntropyLoss: {pesos_classe.tolist()}')
print('Salve este tensor e passe como parâmetro `weight` ao instanciar nn.CrossEntropyLoss().')

## 7. Resumo

- **Integridade:** Todas as imagens foram verificadas.
- **Augmentações:** Flip, rotação, ColorJitter e affine transform aplicados apenas no treino.
- **Normalização:** Estatísticas ImageNet — compatível com Transfer Learning.
- **Desbalanceamento:** Pesos por classe calculados — usar `weight` no `CrossEntropyLoss`.
- **DataLoaders:** Funcionando corretamente, shape `(N, 3, 224, 224)`.

**Próximo passo:** `03_treinamento.ipynb` — treinar o modelo com Transfer Learning.